# CNV overlap with schizophrenia and other disorders


Run this notebook separately on **PCNET2.2** (default) and **STRING_high_conf** by setting `CNV_NETWORK`; `sbatch jobs/job.sh` runs both.
Executed notebooks are `executed_pcnet2.2_<notebook>.ipynb` and `executed_string_<notebook>.ipynb`.
Results, figures and reports are separated under `results/PCNET2.2/` and `results/STRING_high_conf/`; propagation caches under `data/cache/PCNET2.2/` and `data/cache/STRING_high_conf/`.
Completed results are content-addressed and reused, not regenerated, when inputs, settings and software are unchanged.
STRING uses human high-confidence (combined score ≥ 700) physical interactions, cached with frozen unambiguous Entrez mappings. Each run recomputes its network universe, seed QC and network-specific nulls.
Gene sets with fewer than `MIN_SEEDS` (3) seeds inside the selected network are excluded from that network's run.


In [ ]:
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cnv_overlap.pipeline import AnalysisConfig, run
from cnv_overlap.visualization import pair_heatmap, size_scatter


ROOT = Path.cwd()
if ROOT.name == "notebooks":  # opened from notebooks/ in Jupyter
    ROOT = ROOT.parent
if not (ROOT / "configs").is_dir():
    raise RuntimeError("Start this notebook with the project root as working directory")
CONFIG_PATH = Path(os.environ.get("CNV_CONFIG", ROOT / "configs/cnv_disorder_overlap.json"))
import sys
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
from network_inputs import load_network_inputs
# Set CNV_NETWORK=STRING_high_conf for the independent STRING sensitivity run.
NETWORK = os.environ.get("CNV_NETWORK", None if "CNV_CONFIG" in os.environ else "PCNET2.2")
# Minimum seeds a gene set needs inside the selected network; smaller sets are excluded.
MIN_SEEDS = 3
spec, root, inputs, provenance = load_network_inputs(CONFIG_PATH, NETWORK, min_seeds=MIN_SEEDS)
RESULTS_ROOT = (root / spec["output_root"]).parent
print("Network:", NETWORK or "custom snapshot", "Results:", RESULTS_ROOT, "Cache:", root / spec["cache_dir"])
config = AnalysisConfig.from_dict(spec["analysis"])
if config.comparisons != "cnv_disorder":
    raise ValueError("This notebook requires analysis.comparisons=cnv_disorder")
A, nodes, symbols, entities, coordinates, regions = inputs
print(f"{len(nodes):,} nodes, {A.nnz // 2:,} edges; {len(entities)} gene sets")
print("Primary: overlap size; NetColoc normal-tail p-values; overlap policy:", config.overlap)
print("Gene-label permutations:", config.permutation_reps)

from notebook_analysis import SCZ_TARGETS, disorder_gene_partition


In [ ]:
# One run gives one BH family across every CNV–target pair for each statistic.
entities = {name: entity for name, entity in entities.items() if name != "t1d"}
if not any(e["kind"] == "cnv" for e in entities.values()) or not any(e["kind"] == "target" for e in entities.values()):
    raise ValueError("Need CNVs and at least one target set")
HAS_SCZ = any(name in entities and entities[name]["kind"] == "target" for name in SCZ_TARGETS)
FAMILY_INPUTS = {"combined": (A, nodes, symbols, entities, coordinates, regions)}
RUNS = {"combined": run(*FAMILY_INPUTS["combined"], config=config,
    output_root=root/spec["output_root"]/"combined", cache_dir=root/spec["cache_dir"],
    provenance={**provenance, "question_family": "combined",
                "excluded_targets": ["t1d"],
                "analysis_status": "exploratory revision after earlier results"})}
for family, run_dir in RUNS.items():
    assert json.loads((run_dir / "manifest.json").read_text())["status"] == "complete"
    print(family, run_dir)
    print((run_dir / "summary.md").read_text())
from cnv_overlap.provenance import digest
FIGURES = RESULTS_ROOT / "figures/disorder_structure" / digest({k: v.name for k, v in RUNS.items()})[:20]
FIGURES.mkdir(parents=True, exist_ok=True)
if not HAS_SCZ:
    print("No schema/gwaspri sets: these custom inputs have secondary comparisons only.")


## Input QC and tested hypotheses

Inspect seed losses, coordinate coverage and the hypothesis list. Coordinates document CNV content; all retained network genes are eligible for degree-matched references.
The SCZ GWAS prioritized genes and comparator Catalog genes have different ascertainment; see `input/processed/source_manifest.json`.


In [ ]:
for family, run_dir in RUNS.items():
    print("Family:", family)
    display(pd.read_csv(run_dir / "seed_qc.tsv", sep="\t"))
    display(pd.read_csv(run_dir / "seed_overlap.tsv", sep="\t").set_index("entity"))
    print("Planned comparisons:", len(pd.read_csv(run_dir / "hypotheses.tsv", sep="\t")))


## B. Combined schizophrenia and other disorder profiles

One BH discovery family covers all 81 default CNV–target tests, separately for each null and statistic. SCHEMA and GWAS-prioritized evidence remain distinct targets. T1D is excluded; counts adapt to custom inputs.

Show every pair, including negative or nonsignificant excess. Stars annotate combined-family BH q < 0.05 for the size endpoint. `p`/`q` use the existing normal-tail method; `q_empirical` applies the same combined correction to empirical p-values as a sensitivity diagnostic. Each pair retains its own permutation null. Neither correction nor display constitutes a direct comparison between disorders. Permutation-tail intervals are not effect-size confidence intervals.


In [ ]:
from cnv_overlap.statistics import bh
frames = []
for family, run_dir in RUNS.items():
    frame = pd.read_csv(run_dir / "pair_tests.tsv", sep="\t").assign(question_family=family)
    frame["target_group"] = np.where(frame.b.isin(SCZ_TARGETS), "scz", "secondary")
    frame["q_empirical"] = np.nan
    for _, idx in frame.groupby(["null", "family", "statistic"]).groups.items():
        frame.loc[idx, "q_empirical"] = bh(frame.loc[idx, "p_empirical"])
    frames.append(frame)
    for metric in ("ratio", "excess", "observed", "z"):
        fig = pair_heatmap(run_dir, family="locus_target", fdr=config.fdr, metric=metric)
        fig.savefig(FIGURES / f"{family}_{metric}.png", dpi=200, bbox_inches="tight")
        fig.savefig(FIGURES / f"{family}_{metric}.pdf", bbox_inches="tight")
        plt.show()
    # Non-schizophrenia disorders only: z-score colour, colocalised-gene counts as labels.
    fig = pair_heatmap(run_dir, family="locus_target", fdr=config.fdr, metric="z", exclude=SCZ_TARGETS)
    fig.savefig(FIGURES / f"{family}_z_no_scz.png", dpi=200, bbox_inches="tight")
    fig.savefig(FIGURES / f"{family}_z_no_scz.pdf", bbox_inches="tight")
    plt.show()
    fig = size_scatter(run_dir, family="locus_target", fdr=config.fdr)
    fig.savefig(FIGURES / f"{family}_sizes.png", dpi=200, bbox_inches="tight")
    fig.savefig(FIGURES / f"{family}_sizes.pdf", bbox_inches="tight")
    plt.show()
tests = pd.concat(frames, ignore_index=True)
primary = tests[(tests["null"] == "netcoloc") & (tests.statistic == "size")].copy()
primary.to_csv(FIGURES / "all_primary_pairs.tsv", sep="\t", index=False)
with pd.option_context("display.max_rows", None):
    display(primary.sort_values(["question_family", "a", "b"])[["question_family", "a", "b",
        "observed", "null_mean", "excess", "p", "q", "p_empirical", "q_empirical", "status"]])


## All-pair gene candidates and descriptive recurrence

Candidate genes meet fixed joint-score thresholds. All candidates are retained, irrespective of pair q-value. Recurrence below counts CNVs and targets among these thresholded pairs; it has no gene-level significance claim. The multi-CNV notebook tests convergence independently of this pair-selection summary.

In [ ]:
display(tests[tests.statistic != "size"].sort_values(["question_family", "statistic", "q"]).head(30))
candidates = pd.concat([pd.read_csv(p / "candidate_genes.tsv", sep="\t").assign(question_family=f)
                        for f, p in RUNS.items()], ignore_index=True)
candidates.to_csv(FIGURES / "all_candidate_genes.tsv", sep="\t", index=False)
display(candidates.head(40))
recurrence = candidates.groupby(["entrez", "symbol"]).agg(n_pairs=("a", "size"),
    n_cnvs=("a", "nunique"), n_targets=("b", "nunique")).reset_index()
recurrence.to_csv(FIGURES / "all_pair_descriptive_recurrence.tsv", sep="\t", index=False)
display(recurrence.sort_values(["n_cnvs", "n_pairs"], ascending=False).head(30))


## Descriptive communities

Each connected component is partitioned separately. Distinct components are never pooled into a fictitious community zero. Recurrence, degree and community membership describe the candidate network; they are not independent replication or significance tests.

In [ ]:
from cnv_overlap.interpretation import candidate_network
G, membership = candidate_network(A, nodes, candidates)
membership["symbol"] = membership.entrez.map(symbols)
membership.to_csv(FIGURES / "candidate_communities.tsv", sep="\t", index=False)
print(f"{G.number_of_nodes()} candidate nodes, {G.number_of_edges()} edges")
display(membership.sort_values(["community", "degree_in_candidate_network"], ascending=[True, False]).head(40))


## Offline annotation and rewiring robustness

Run annotation with explicit GMT snapshots using `cnv-overlap annotate`; query/background use the same selected network universe. Enrichment is descriptive ORA after network selection. Run `cnv-overlap rewire --config configs/cnv_disorder_overlap.json` to rescore independent chain endpoints with graph-specific propagation references. Chain trajectories diagnose mixing; no p-value is inferred from three endpoints. These separate outputs do not silently alter the primary run.

In [ ]:
from notebook_analysis import export_versioned_report
GMT_PATHS = [root / "input/annotations" / name for name in
             ["GO_Biological_Process_2026.gmt", "ReactomePathways.gmt"]]
if not all(path.exists() for path in GMT_PATHS):
    print("Annotation snapshots missing; exporting figures and networks only.")
    GMT_PATHS = []
REPORTS = {}
for family, run_dir in RUNS.items():
    REPORTS[family] = export_versioned_report(run_dir, FAMILY_INPUTS[family],
        output_dir=RESULTS_ROOT / "reports" / family / run_dir.name,
        gmt_paths=GMT_PATHS, fdr=config.fdr)
    print(family, REPORTS[family])
    if GMT_PATHS:
        annotation = pd.read_csv(REPORTS[family] / "enrichment/descriptive_ora.tsv", sep="\t")
        selected_pairs = primary.loc[(primary.question_family == family) & (primary.q < config.fdr), ["a", "b"]]
        displayed = annotation.merge(selected_pairs, on=["a", "b"])
        display(displayed[displayed.q_all_pair_terms < config.fdr].sort_values("q_all_pair_terms").head(40))


## Shared SCZ and additional disorder footprints within each CNV

In [ ]:
partitioned, partition_summary = disorder_gene_partition(candidates, primary)
if not HAS_SCZ:
    print("SCZ partition unavailable: no SCZ target sets were supplied.")
else:
    partitioned.to_csv(FIGURES / "secondary_gene_partitions.tsv", sep="\t", index=False)
    partition_summary.to_csv(FIGURES / "secondary_partition_counts.tsv", sep="\t", index=False)
    with pd.option_context("display.max_rows", None):
        display(partition_summary)
FOCAL_CNV = "22q11.2"
if FOCAL_CNV not in set(primary.a):
    FOCAL_CNV = sorted(primary.a.unique())[0]
print("Focal CNV:", FOCAL_CNV)
display(primary[primary.a == FOCAL_CNV].sort_values(["question_family", "b"]))
if GMT_PATHS and HAS_SCZ and not partitioned.empty:
    from cnv_overlap.enrichment import load_gmt, ora
    terms = {}
    for path in GMT_PATHS:
        for term, definition in load_gmt(path).items():
            if term in terms and terms[term] != definition:
                raise ValueError(f"Conflicting annotation term: {term}")
            terms[term] = definition
    background = {symbols[int(g)] for g in nodes}
    ora_frames = []
    for target in sorted(primary.loc[(primary.a == FOCAL_CNV) & (primary.target_group == "secondary"), "b"]):
        for category in ("shared_with_scz", "secondary_only_at_threshold"):
            query = partitioned[(partitioned.a == FOCAL_CNV) & (partitioned.b == target) &
                                (partitioned.category == category)].symbol.dropna()
            frame = ora(query, background, terms).assign(cnv=FOCAL_CNV, target=target, category=category)
            ora_frames.append(frame)
    if ora_frames:
        focal_ora = pd.concat(ora_frames, ignore_index=True)
        focal_ora["q_all_focal_terms"] = bh(focal_ora.p)
        focal_ora.to_csv(FIGURES / "focal_partition_descriptive_ora.tsv", sep="\t", index=False)
        display(focal_ora.sort_values("q_all_focal_terms").head(40))
